# 00 · See, calculate, explain

**Mission:** explore a browser EOC and global health teaching environment. In 15 minutes, identify data provenance, run a calculation, inspect a map, and download a reusable layer.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then Run → Run All Cells. First use downloads Matplotlib. All training data are **SYNTHETIC** unless explicitly labeled LIVE. Downloads appear below and in `exports/`; the browser filesystem is separate from your computer. Each lab runs independently.

In [1]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("matplotlib")
import json, math
from pathlib import Path
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from worldview_lab import (load, export, map_layer, chart_style, haversine_km,
    wilson_interval, shortest_times, freshness, normalize_quakes,
    validate_geojson, brief_request, validate_brief)
chart_style()

## Choose a route

| Route | Labs | Outcome |
|---|---|---|
| Map foundations | 00–03 | Live-feed quality, dashboard literacy, validated GeoJSON |
| EOC + public health | 04–06 | Hazard proximity, outbreak rates, access under road closures |
| 3D + AI | 07–09 | Animated scene, Astra evidence briefing, raster change interpretation |
| Capstone | 10 | A defensible briefing with reproducible evidence |

[Open the 2D EOC](https://jltobias.github.io/JupyterLite-WorldView/dashboard/operations.html) · [Open the 3D scene](https://jltobias.github.io/JupyterLite-WorldView/scenes/) · [Read the book](https://jltobias.github.io/JupyterLite-WorldView/book/)

In [2]:
manifest=load("manifest.json")
display(manifest)

{'version': 1,
 'generated_by': 'scripts/generate_data.py',
 'license': 'MIT',
 'classification': 'SYNTHETIC — no real districts, facilities, cases, flight or roads',
 'scenario_clock': '2026-09-14T12:00:00Z',
 'crs': 'WGS 84 longitude, latitude (degrees)',
 'population_definition': 'Fictional resident population, fixed over a 14-day interval',
 'cases_definition': 'Fictional new cases by onset date, one per person, over 2026-09-01 through 2026-09-14',
 'geography': 'Illustrative grid near Washington, DC; not administrative boundaries',
 'caveats': ['No operational or clinical use',
  'Reporting fractions are invented sensitivity assumptions',
  '3D heights encode rates, not terrain or buildings',
  'No personal or individual-level health data']}

In [3]:
districts=load('districts.geojson')
population=sum(f['properties']['population'] for f in districts['features'])
cases=sum(f['properties']['cases'] for f in districts['features'])
print(f'{len(districts["features"])} fictional districts | population {population:,} | 14-day cases {cases}')
print(f'Pooled 14-day new cases per 100,000: {cases/population*100000:.1f}')
map_layer(districts)

12 fictional districts | population 203,500 | 14-day cases 477
Pooled 14-day new cases per 100,000: 234.4


<iframe title="SYNTHETIC teaching layer" sandbox="allow-scripts" style="width:100%;height:470px;border:0" srcdoc="<!doctype html><html lang="en"><meta charset="utf-8"><link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
 <style>body{margin:0;font:14px system-ui}#map{height:420px}header{padding:10px;background:#102536;color:#fff}</style>
 <header>SYNTHETIC teaching layer | cyan &lt;200 · amber 200–399 · coral ≥400 per 100k (when rate is selected)</header><div id="map"></div>
 <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script><script>
 const data={"type": "FeatureCollection", "features": [{"type": "Feature", "id": "D01", "properties": {"id": "D01", "name": "Exercise district 01", "population": 8000, "cases": 12, "rate_per_100k": 150.0, "reporting_fraction": 1.0, "longitude": -77.235, "latitude": 38.71, "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T12:00:00Z", "status": "SIMULATED", "height_m": 750}, "geometry": {"type": "Polygon", "coordinates": [[[-77.3, 38.65], [-77.17999999999999, 38.65], [-77.17999999999999, 38.76], [-77.3, 38.76], [-77.3, 38.65]]]}}, {"type": "Feature", "id": "D02", "properties": {"id": "D02", "name": "Exercise district 02", "population": 18000, "cases": 27, "rate_per_100k": 150.0, "reporting_fraction": 0.85, "longitude": -77.105, "latitude": 38.71, "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T12:00:00Z", "status": "SIMULATED", "height_m": 750}, "geometry": {"type": "Polygon", "coordinates": [[[-77.17, 38.65], [-77.05, 38.65], [-77.05, 38.76], [-77.17, 38.76], [-77.17, 38.65]]]}}, {"type": "Feature", "id": "D03", "properties": {"id": "D03", "name": "Exercise district 03", "population": 12500, "cases": 40, "rate_per_100k": 320.0, "reporting_fraction": 0.7, "longitude": -76.975, "latitude": 38.71, "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T12:00:00Z", "status": "SIMULATED", "height_m": 1600}, "geometry": {"type": "Polygon", "coordinates": [[[-77.03999999999999, 38.65], [-76.91999999999999, 38.65], [-76.91999999999999, 38.76], [-77.03999999999999, 38.76], [-77.03999999999999, 38.65]]]}}, {"type": "Feature", "id": "D04", "properties": {"id": "D04", "name": "Exercise district 04", "population": 24000, "cases": 32, "rate_per_100k": 133.33333333333334, "reporting_fraction": 1.0, "longitude": -76.845, "latitude": 38.71, "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T12:00:00Z", "status": "SIMULATED", "height_m": 667}, "geometry": {"type": "Polygon", "coordinates": [[[-76.91, 38.65], [-76.78999999999999, 38.65], [-76.78999999999999, 38.76], [-76.91, 38.76], [-76.91, 38.65]]]}}, {"type": "Feature", "id": "D05", "properties": {"id": "D05", "name": "Exercise district 05", "population": 9000, "cases": 45, "rate_per_100k": 500.0, "reporting_fraction": 0.85, "longitude": -77.235, "latitude": 38.83, "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T12:00:00Z", "status": "SIMULATED", "height_m": 2500}, "geometry": {"type": "Polygon", "coordinates": [[[-77.3, 38.769999999999996], [-77.17999999999999, 38.769999999999996], [-77.17999999999999, 38.879999999999995], [-77.3, 38.879999999999995], [-77.3, 38.769999999999996]]]}}, {"type": "Feature", "id": "D06", "properties": {"id": "D06", "name": "Exercise district 06", "population": 30000, "cases": 48, "rate_per_100k": 160.0, "reporting_fraction": 0.7, "longitude": -77.105, "latitude": 38.83, "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T12:00:00Z", "status": "SIMULATED", "height_m": 800}, "geometry": {"type": "Polygon", "coordinates": [[[-77.17, 38.769999999999996], [-77.05, 38.769999999999996], [-77.05, 38.879999999999995], [-77.17, 38.879999999999995], [-77.17, 38.769999999999996]]]}}, {"type": "Feature", "id": "D07", "properties": {"id": "D07", "name": "Exercise district 07", "population": 15000, "cases": 90, "rate_per_100k": 600.0, "reporting_fraction": 1.0, "longitude": -76.975, "latitude": 38.83, "source": "SYNTHETIC exercise v1", "

## Observe → calculate → explain → review

The map locates differences. Python calculates them. Astra can help inspect code, interpret an exported figure, and draft a cited briefing. A reviewer checks all three. Astra does not run inside Pyodide: the optional API runner executes on your computer or a controlled server, with credentials outside the static site.

**Try:** compare the biggest district case count with the highest population-normalized value. Why are these different questions? **Checkpoint:** a pooled measure divides total cases by total population; averaging district rates gives each district equal weight regardless of size.

In [4]:
p=[f['properties'] for f in districts['features']]
display({'most_cases':max(p,key=lambda r:r['cases'])['id'],
         'highest_rate':max(p,key=lambda r:r['rate_per_100k'])['id']})
export('districts.geojson',districts)

{'most_cases': 'D07', 'highest_rate': 'D07'}

WindowsPath('exports/districts.geojson')

## Sources and attribution

[JupyterLite](https://jupyterlite.readthedocs.io/en/stable/) explains the browser runtime. [GPT-6 Astra](https://developers.openai.com/api/docs/models/gpt-6-astra) documents model features; this curriculum applies them to GIS workflows.

World View inspiration: [Kevin (Khoa) To, MIT](https://github.com/kevtoe/worldview). Original lab code, figures, and synthetic fixtures: JupyterLite WorldView contributors, MIT. See the [book references](https://jltobias.github.io/JupyterLite-WorldView/book/references.html) and repository notices for dependency and service terms.